# Using Claude to generate the Verhaal Speciaal story

** Code has been tested with Python 3.12 **

In this notebook we will create a 'Verhaal Speciaal' story.

The story will be generated by a LLM (GPT-4) based on a prompt. 

The story will be based on user input. One of the user inputs is the reading level, which is based on class/group. 

In the last part of the notebook we can evaluate the level of the generated text.

### Contents
0. Installs and imports
1. Settings and prompt
2. Generate chapter one
3. Generate chapter two & three
4. Convert story to JSON

## 0. Installs and imports

In [487]:
#!pip install openai --upgrade

In [488]:
!which python3

/opt/anaconda3/bin/python3


In [489]:
import anthropic
anthropic.__version__

'0.51.0'

In [490]:
#import the local files
import config
import leesniveaus

## 1. Settings

### Setting the reading levels
Four different reading levels have been defined, see below.

Both characters have their own reading level as Verhaal Speciaal is meant to be a reading combination for parent and child. Example: parent can have reading level 3, while the child can have reading level 1.  All combinations are possible. 

### User input

The user_input prompt collects the input the user of the story creates. This is taken from the javascript code of the original Verhaal Speciaal:
1. personage een        => character_one
2. personage twee       => character_two
3. wat                  => plot
4. waarom               => reasoning
5. waar                 => setting
6. wanneer              => time

And we set the reading levels:
7. klas/groep           => groep (will be mapped to reading_level)
8. Leesniveau ouder     => reading level

### Building the prompt 

The reading levels and user input are then used to build up a prompt..

We will generate a prompt consisting of three 'sub-prompts':

prompt =  basic_prompt + previous_text + chapter_prompt 

**basic_prompt**

The basic prompt sets the structure of the story. It defines there are two characters and a story teller.  It makes sure the story follows a pattern.  

**previous_text**

Only used for chapters 2 and 3. 
Input of the previous chapter(s): one or two. 

**chapter_prompt**

This are the chapter specific inputs:

1. Start new story, introduce characters, leave room for chapters 2 and 3
2. Follow up on chapter 1, use previous text and leave room for chapter 3
3. chapter 3: Final chapter, end the story, use previous text.

### Reading levels

In [491]:
#variables based on the reading level settings
level_one = leesniveaus.level_one
level_two = leesniveaus.level_two
level_three = leesniveaus.level_three
level_four = leesniveaus.level_four


### User input

In [492]:
#These are the variables from the front end about the story . 
character_one = 'eddy'
character_two = 'jan'
plot ='een wandeling'
reasoning = 'ze verdwalen'
setting = 'in het bos' #waar
time = 'in de zomer'

# These are the input variables from the front end for the reading level
group_child = 7 #class the child is in 3,4,5,6,7,8
reading_level_parent = level_four # 1 2 3 4 based on reading level settings 

In [493]:
#Reading level conversion table CHILD

group = group_child #class the child is in 3,4,5,6,7,8

if group < 4:
    reading_level_child = level_one
    print(reading_level_child)
elif group == 4:
    reading_level_child = level_two
    print(reading_level_child)
elif group <= 6:
    reading_level_child = level_three
    print(reading_level_child)
elif group <= 8:
    reading_level_child = level_four
    print(reading_level_child)


Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.



In [494]:
#summarizing the reading levels
print(f'Reading level child: {reading_level_child[11:12]}')
print(f'Reading level parent: {reading_level_parent[11:12]}')

Reading level child: 4
Reading level parent: 4


### basic_prompt

In [495]:
#update reading levels
basic_prompt_v4 =f'''Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is {reading_level_parent}.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage {character_one}.
Het leesniveau van personage {character_one} is niveau {reading_level_child}, dus houd het taalgebruik op dat niveau voor dit personage. Gebruik hiervoor de omschrijving van de hiervoor genoemde niveuas
Dit is een beschrijving van personage {character_two}.
Het leesniveau van personage {character_two} is niveau {reading_level_parent}, dus houd het taalgebruik op dat niveau voor dit personage. Gebruik hiervoor de omschrijving van de hiervoor genoemde niveaus houdt het hoofdstuk bij twee zinnen per karakter.

De algemene verhaallijn is: {plot}.
Dit is de reden achter het verhaal: {reasoning}.
De setting van het verhaal is: {setting}.
De tijd waarin het verhaal zich afspeelt is: {time}.

Gebruik de volgende regels om te output te structureren:
Iedere zin of paragraaf van het verhaal moet bij de Verteller, {character_one} of {character_two} horen. 
De verteller wordt altijd aangeduid als Verteller. Gebruik het format Verteller | tekst
Voeg geen code tussen haakjes toe voor de Verteller.

Als een personage wat gaat vertellen, voeg {{char1}} of {{char2}} toe voor de naam van het personage dat spreekt.
voeg een | tussen alle woorden in zoals in dit voorbeeld: {{char1}} | {character_one} | tekst.
Aan het einde van het hoofdstuk moet de verteller een vraag stellen aan een van de personages over de voorgaande dialoog.
Aan het einde van het hoofdstuk moet de tekst '''"{ENDOFACT}"''' op een nieuwe regel worden toegevoegd.
Begin het hoofdstuk duidelijk met het nummer van het hoofdstuk. Bijvoorbeeld: 'Hoofdstuk 1'.
Zorg ervoor dat de personages hetzelfde blijven in de verschillende hoofdstukken en dat ze weten wat er gezegd is.
Voeg geen uitleg toe, alleen de dialoog.
Voeg geen nieuwe personages of settings toe aan de dialoog.
De allereerste regel van de tekst moet een gegenereerde titel zijn. Gebruik alleen letters en spaties, in de titel staat niet het woord 'titel'.
Gebruik geen speciale tekens in de tekst, alleen letters, spaties en nieuwe regels.
'''

### Previous text

In [496]:
#for chapter one empty, for chapter 2/3 will be updated, see below
previous = " " 

### Chapter prompts

In [497]:
chapter_1 = f'''Dit is het eerste hoofdstuk van drie, zorg dus dat het verhaal verder kan gaan.'''
chapter_2 = f'''Dit is het tweede hoofdstuk van drie. Ga door op het eerste hoofdstuk wat je uit deze tekst haalt: {previous}. Zorg dat het verhaal verder kan gaan in hoofdstuk 3. Begin de tekst met de titel van het verhaal en dan Hoofdstuk 2 '''
chapter_3 = f'''Dit is het laatste hoofdstuk dus zorg voor een goed en happy einde. Ga door met het verhaal gebaseerd op hoofdstuk 1 en 2 wat je uit de deze tekst haalt: {previous}. Begin de tekst met de titel van het verhaal en dan Hoofdstuk 3.'''

### Concatenate prompt
We will use v3 as this is the 3rd version of the prompt (to keep it similar to original Verhaal Speciaal)

prompt_v3 = basic_prompt + chapter_prompt

In [498]:
prompt_v4_ch1 = basic_prompt_v4 + chapter_1 
print(prompt_v4_ch1)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage eddy.
Het leesniveau van personage eddy is niveau Leesniveau 4: 
Samengestelde zinnen komen voor. 

## 2. Generate chapter one

In [499]:
import anthropic
import config

client = anthropic.Anthropic(
    # defaults to os.environ.get("ANTHROPIC_API_KEY")
    api_key=config.ANTHROPIC_API_KEY,
)
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt_v4_ch1}
    ]
)
print(message.content[0])

TextBlock(citations=None, text='Verdwaald in het Zomerbos\n\nHoofdstuk 1\n\nVerteller | De zomerzon schijnt helder door de groene bladeren van het bos. Eddy en Jan wandelen over het smalle bospad dat kronkelt tussen de hoge bomen. De vogels fluiten vrolijk en de bijen zoemen rond de wilde bloemen.\n\n{char1} | eddy | Wat een prachtige dag voor een wandeling, vind je niet Jan?\n\n{char2} | jan | Absoluut fantastisch! De natuur is hier zo mooi en rustig.\n\n{char1} | eddy | Ik denk dat we al een uur lopen, maar ik herken dit pad helemaal niet meer.\n\n{char2} | jan | Nu je het zegt, dit deel van het bos lijkt me totaal onbekend voor.\n\nVerteller | De jongens kijken om zich heen en zien alleen maar dichte begroeiing. Het pad dat ze gevolgd hebben wordt steeds smaller en onduidelijker. Welke richting moeten ze nu kiezen, Eddy?\n\n{ENDOFACT}', type='text')


In [500]:
#Function to call the Anthropic API
def create_chat_completion(prompt, model="claude-sonnet-4-20250514"):
  
    message = client.messages.create(
        model= model,
        max_tokens=1024,
        messages=[
            {"role": "user", 
             "content": prompt}
        ]   
    )

    # Return the generated response
    chapter = message.content[0].text
    return chapter


In [501]:
chapter_one = create_chat_completion(prompt_v4_ch1)
print(chapter_one)

De Verdwaalde Wandelaars

Hoofdstuk 1

Verteller | De zomerse middag was ideaal voor een wandeling door het uitgestrekte bos. Eddy en Jan liepen enthousiast over het kronkelende bospad, terwijl de zonnestralen door de bladeren dansten.

{char1} | eddy | Wat een prachtige dag voor een wandeling, Jan! Het bos ruikt zo fris en natuurlijk.

{char2} | jan | Absoluut waar, Eddy! De vogels zingen zo melodieus en de temperatuur is perfect voor onze expeditie.

Verteller | Na een tijdje kwamen ze bij een splitsing waar meerdere paden verschillende richtingen opgingen. De bordjes waren helaas door de tijd onleesbaar geworden.

{char1} | eddy | Hmm, welke route zullen we kiezen? Links gaat het pad omhoog naar de heuvels, rechts lijkt het meer naar het dal te leiden.

{char2} | jan | Laten we het linkerpad proberen, dat ziet er avontuurlijker uit! Bovendien kunnen we vanaf de heuvel vast een prachtig panorama bewonderen.

Verteller | Eddy, waarom vond je het linkerpad eigenlijk een goede keuze?

{

## 3. Generate chapter two & three

The first chapter is input for chapter two and three. 



In [502]:
previous = f"Het vorige hoofdstuk was: {chapter_one}."

In [503]:
prompt_v4_ch2 = basic_prompt_v4+previous+chapter_2
print(prompt_v4_ch2)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage eddy.
Het leesniveau van personage eddy is niveau Leesniveau 4: 
Samengestelde zinnen komen voor. 

In [504]:
chapter_two = create_chat_completion(prompt_v4_ch2)
print(chapter_two)

De Verdwaalde Wandelaars

Hoofdstuk 2

Verteller | Na een uur lopen door dichte vegetatie en kronkelende paadjes, realiseerden Eddy en Jan zich dat ze de weg volledig kwijt waren. De bekende geluiden van de hoofdroute waren verdwenen.

{char1} | eddy | Jan, ik herken deze omgeving helemaal niet meer! We hadden allang bij de picknickplaats moeten zijn.

{char2} | jan | Je hebt gelijk, Eddy. Deze ruige paden lijken steeds smaller te worden en ik zie geen enkele wegwijzer meer.

Verteller | De zon begon langzaam te zakken en wierp lange schaduwen tussen de bomen. Het bos werd stiller en de jongens voelden zich steeds onzekerder over hun situatie.

{char1} | eddy | Misschien moeten we teruglopen naar die splitsing? Hoewel ik niet zeker weet of we die nog kunnen vinden.

{char2} | jan | Dat is een rationele gedachte! Of we kunnen proberen een hoge boom te klimmen om het landschap te overzien.

Verteller | Jan, denk je echt dat klimmen in een boom jullie uit deze situatie kan helpen?

{ENDOF

In [505]:
#calling chapter trhee
previous = f"De vorige hoofdstukken waren {chapter_one} en {chapter_two}."
prompt_v4_ch3 = basic_prompt_v4+previous+chapter_3
print(prompt_v4_ch3)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage eddy.
Het leesniveau van personage eddy is niveau Leesniveau 4: 
Samengestelde zinnen komen voor. 

In [506]:
chapter_three = create_chat_completion(prompt_v4_ch3)
print(chapter_three)

De Verdwaalde Wandelaars

Hoofdstuk 3

Verteller | Jan klom behendig in een stevige eikenboom en keek over de uitgestrekte groene zee van bladeren. Plotseling zag hij in de verte een bekende toren die ze eerder hadden gezien.

{char2} | jan | Eddy, ik zie de oude watertoren! Die stond vlakbij het startpunt van onze wandeling, dus we moeten die richting opgaan.

{char1} | eddy | Fantastisch, Jan! Nu weet ik weer waar we zijn, want die toren is inderdaad ons oriëntatiepunt geweest.

Verteller | Vol nieuwe energie volgden ze Jans aanwijzingen en wandelden doelbewust door het bos. Na een half uur kwamen ze eindelijk bij het vertrouwde bospad uit.

{char2} | jan | Gelukkig zijn we weer op de juiste route! Onze avontuurlijke omweg heeft ons veel geleerd over navigatie.

{char1} | eddy | Wat een spannende ervaring was dit! Volgende keer nemen we zeker een kompas mee voor onze bosexpedities.

Verteller | Eddy, wat vond je het belangrijkste wat jullie vandaag hebben geleerd?

{ENDOFACT}


## 4. Save story as JSON

We need to create a .json file out of this story. This .json file will be analysed by the other scripts in the 'AVI Score' repository. 

The JSON structure is relatively simple, containing just one key-value pair. The complexity lies in the structured text content rather than in nested JSON objects or arrays.

The JSON file contains a single object with one key-value pair:
Key: "text"
Value: A long string containing a story

- First line contains the title
- Narrator sections: Paragraphs starting with "Verteller |"
- Character dialogues: Lines starting with "{char1} |" or "{char2} |"

Character dialogues follow this pattern:
- {char1} | anna | [dialogue text]
- {char2} | tom | [dialogue text]

Other things
- "{ENDOFACT}" appears at the end, likely indicating the end of a story act or section.
- The text uses newline characters (\n) to separate lines and sections.
- There are no nested objects or arrays within this JSON structure.
- The entire story is contained within a single string value.


In [507]:
story = {"text": chapter_one + chapter_two + chapter_three}
print(story['text'])

De Verdwaalde Wandelaars

Hoofdstuk 1

Verteller | De zomerse middag was ideaal voor een wandeling door het uitgestrekte bos. Eddy en Jan liepen enthousiast over het kronkelende bospad, terwijl de zonnestralen door de bladeren dansten.

{char1} | eddy | Wat een prachtige dag voor een wandeling, Jan! Het bos ruikt zo fris en natuurlijk.

{char2} | jan | Absoluut waar, Eddy! De vogels zingen zo melodieus en de temperatuur is perfect voor onze expeditie.

Verteller | Na een tijdje kwamen ze bij een splitsing waar meerdere paden verschillende richtingen opgingen. De bordjes waren helaas door de tijd onleesbaar geworden.

{char1} | eddy | Hmm, welke route zullen we kiezen? Links gaat het pad omhoog naar de heuvels, rechts lijkt het meer naar het dal te leiden.

{char2} | jan | Laten we het linkerpad proberen, dat ziet er avontuurlijker uit! Bovendien kunnen we vanaf de heuvel vast een prachtig panorama bewonderen.

Verteller | Eddy, waarom vond je het linkerpad eigenlijk een goede keuze?

{

In [508]:
import json
import os
from datetime import datetime

# Convert the string into a JSON serializable format, e.g., as a dictionary
story_to_save = story

# Get the current date
current_date = datetime.now().strftime("%Y-%m-%d_%H:%M")

# Create a filename with the current date
filename = f"vs_claude_rl{reading_level_child[11:12]}_{current_date}.json"
file_path = os.path.join('json', filename)

# Save the data to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(story_to_save, json_file, indent=4)

print(f"Data saved to {filename}")

Data saved to vs_claude_rl4_2025-06-02_10:26.json


## 5. Validate the new .json with an old example

In [509]:
#Here we use an old .json based on the Javascript code base

import json
from pprint import pprint

# Read the JSON file
with open('./json/V_S_2025-05-27_12:20.json', 'r') as file:
    data = json.load(file)

# Pretty print using json.dumps()
print("Pretty printed using json.dumps():")
print(json.dumps(data, indent=4))

# Pretty print using pprint
print("\nPretty printed using pprint:")
pprint(data['text'])

Pretty printed using json.dumps():
{
    "text": "Verdwaald in het Groene Bos\n\nHoofdstuk 1\n\nVerteller | De zomerzon schijnt helder door de bladeren van het dichte bos. Eddy en Jan lopen samen over een smal bospad, terwijl de vogels vrolijk fluiten in de bomen boven hun hoofden.\n\n{char1} | eddy | Wat een prachtige dag voor een wandeling door dit mysterieuze bos!\n\n{char2} | jan | Inderdaad, de natuur is hier heel speciaal en de lucht ruikt zo fris.\n\n{char1} | eddy | Kijk eens naar die kronkelende paden die alle kanten opgaan.\n\n{char2} | jan | Welke route moeten we eigenlijk nemen om terug te komen?\n\nVerteller | Eddy, herinner je je nog welke weg jullie genomen hebben om hier te komen?\n\n{ENDOFACT}De Verdwaalde Wandelaars\n\nHoofdstuk 1\n\nVerteller | Het is een prachtige zomerdag en Eddy en Jan beslissen om een lange wandeling te maken door het dichte bos. De zon schijnt helder tussen de bladeren door en overal horen ze vogels zingen.\n\n{char1} | eddy | Wat een fantastisc

## To do list
- take prompts to config files instead of code

 
